In [2]:
"""
MLB GUMBO API real-time websocket and REST live feed client.
Optimized for execution within an active Jupyter Notebook / ipykernel environment.
"""

import asyncio
import logging
import json
from typing import Any, Dict
import aiohttp
import websockets

# Configure structured logging for Notebook streaming output
# Re-initialize handlers to prevent duplicate lines in notebook cell stdout
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("GumboNotebookClient")

# Test parameters
TEST_GAME_PK = 717325  
PING_INTERVAL_SECONDS = 60
REST_BASE_URL = "https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
WS_BASE_URL = "wss://ws.statsapi.mlb.com/api/v1/game/push/subscribe/gameday/{game_pk}"


async def fetch_gumbo_state(session: aiohttp.ClientSession, game_pk: int, timestamp: str) -> None:
    """
    Queries the REST feed layer using the precise event timestamp parsed
    from the WebSocket push frame to pull down the un-truncated JSON layout.
    """
    url = REST_BASE_URL.format(game_pk=game_pk)
    params = {"timestamp": timestamp}
    
    try:
        async with session.get(url, params=params, timeout=10) as response:
            if response.status == 200:
                data: Dict[str, Any] = await response.json()
                game_data = data.get("gameData", {})
                live_data = data.get("liveData", {})
                linescore = live_data.get("linescore", {})
                
                logger.info(
                    f"Successfully fetched GUMBO payload | "
                    f"Status: {game_data.get('status', {}).get('abstractGameState')} | "
                    f"Inning: {linescore.get('currentInningOrdinal', 'N/A')} | "
                    f"Runs: H {linescore.get('teams', {}).get('home', {}).get('runs', 0)} - "
                    f"A {linescore.get('teams', {}).get('away', {}).get('runs', 0)}"
                )
            else:
                logger.error(f"Failed to fetch GUMBO state payload. HTTP Status: {response.status}")
    except Exception as e:
        logger.error(f"Error occurring during REST fetch task: {str(e)}")


async def send_heartbeat(websocket: websockets.ClientConnection) -> None:
    """
    Looping background task keeping the channel active via the standard 'Gameday5' push frame.
    Updated to use modern websockets.ClientConnection type hinting.
    """
    while True:
        try:
            await asyncio.sleep(PING_INTERVAL_SECONDS)
            logger.debug("Dispatching 'Gameday5' heartbeat sequence...")
            await websocket.send("Gameday5")
        except asyncio.CancelledError:
            break
        except Exception as e:
            logger.error(f"Heartbeat transmitter experienced a connection fault: {str(e)}")
            break


async def stream_gumbo_events(game_pk: int) -> None:
    """
    Main state machine orchestrating socket handshake, ping loop, and event delegation.
    """
    ws_url = WS_BASE_URL.format(game_pk=game_pk)
    logger.info(f"Connecting to WebSocket endpoint: {ws_url}")
    
    async with aiohttp.ClientSession() as http_session:
        try:
            async with websockets.connect(ws_url) as websocket:
                logger.info(f"WebSocket channel established for gamePk: {game_pk}")
                
                # Spin up background heartbeat worker task
                heartbeat_task = asyncio.create_task(send_heartbeat(websocket))
                
                try:
                    async for message in websocket:
                        if message == "Gameday5" or not isinstance(message, str):
                            continue
                        
                        try:
                            payload = json.loads(message)
                            timestamp = payload.get("timeStamp")
                            events = payload.get("gameEvents", [])
                            logical_events = payload.get("logicalEvents", [])
                            
                            logger.info(
                                f"Push Message Received | UpdateID: {payload.get('updateId')} | "
                                f"Events: {events} | Logical: {logical_events}"
                            )
                            
                            if timestamp:
                                asyncio.create_task(fetch_gumbo_state(http_session, game_pk, timestamp))
                                
                        except json.JSONDecodeError:
                            logger.warning(f"Raw frame data could not be parsed to JSON schema: {message}")
                            
                finally:
                    heartbeat_task.cancel()
                    await asyncio.gather(heartbeat_task, return_exceptions=True)
                    
        except Exception as e:
            logger.critical(f"WebSocket connection encountered a fatal error or disconnected: {str(e)}")


# Execution context tailored explicitly for ipykernel's running event loop.
logger.info(f"Initializing testing protocol sequence for game context: {TEST_GAME_PK}")
try:
    # We await the coroutine directly because Jupyter runs inside an active loop.
    await stream_gumbo_events(TEST_GAME_PK)
except asyncio.CancelledError:
    logger.info("Asynchronous task sequence canceled.")

2026-06-15 10:02:04 [INFO] Initializing testing protocol sequence for game context: 717325
2026-06-15 10:02:04 [INFO] Connecting to WebSocket endpoint: wss://ws.statsapi.mlb.com/api/v1/game/push/subscribe/gameday/717325
2026-06-15 10:02:04 [INFO] WebSocket channel established for gamePk: 717325
2026-06-15 10:02:04 [CRITICAL] WebSocket connection encountered a fatal error or disconnected: received 4400 (private use) Game is not available for subscription at this time.; then sent 4400 (private use) Game is not available for subscription at this time.


In [5]:
"""
MLB GUMBO API Historical Data Extraction Script.
Optimized for direct execution within a Jupyter Notebook cell.
"""

import logging
from typing import Any, Dict
import requests

# Re-initialize logging handlers to prevent duplicate rows inside standard notebook output
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("GumboHistoricalClient")

HISTORICAL_GAME_PK = 717325  
HISTORICAL_URL = f"https://statsapi.mlb.com/api/v1.1/game/{HISTORICAL_GAME_PK}/feed/live"


def extract_historical_gumbo(game_pk: int) -> None:
    """
    Queries the final historical GUMBO REST state frame and safely
    unpacks game meta parameters along with deep pitch matrices.
    """
    logger.info(f"Requesting historical GUMBO payload for gamePk: {game_pk}")
    
    try:
        response = requests.get(HISTORICAL_URL, timeout=15)
        
        # Fixed: requests utilizes .status_code instead of aiohttp's .status
        if response.status_code == 200:
            gumbo_data: Dict[str, Any] = response.json()
            
            # 1. Structural Layer: gameData (Teams, Venue, Status, Weather)
            game_data = gumbo_data.get("gameData", {})
            teams = game_data.get("teams", {})
            home_team = teams.get("home", {}).get("name", "Unknown Home")
            away_team = teams.get("away", {}).get("name", "Unknown Away")
            
            # 2. Structural Layer: liveData (Boxscore, Linescore, Intersected Plays)
            live_data = gumbo_data.get("liveData", {})
            linescore = live_data.get("linescore", {})
            
            # Extract cumulative scores safely
            home_runs = linescore.get("teams", {}).get("home", {}).get("runs", 0)
            away_runs = linescore.get("teams", {}).get("away", {}).get("runs", 0)
            
            logger.info("--- GAME METADATA RECOVERY ---")
            logger.info(f"Matchup: {away_team} vs {home_team}")
            logger.info(f"Final Score: {away_team} {away_runs} - {home_team} {home_runs}")
            logger.info(f"Total Innings Tracked: {linescore.get('currentInningOrdinal', 'N/A')}")
            
            # 3. Deep Historical Parse: Extracting structural tracking matrices
            all_plays = live_data.get("plays", {}).get("allPlays", [])
            logger.info(f"Successfully processed {len(all_plays)} complete plate appearances (at-bats).")
            
            if all_plays:
                logger.info("--- SAMPLE AT-BAT DATA EXTRACTION ---")
                sample_play = all_plays[0]  # The initial at-bat sequence of the match
                result = sample_play.get("result", {})
                about = sample_play.get("about", {})
                count = sample_play.get("count", {})
                
                logger.info(f"Inning: {about.get('halfInning', 'N/A').upper()} {about.get('inning', 1)}")
                logger.info(f"Event: {result.get('event')} | Description: {result.get('description')}")
                logger.info(f"Final At-Bat Count: {count.get('balls')}B / {count.get('strikes')}S")
                
                # Sub-array layer containing absolute tracking vectors for every pitch delivered
                pitch_events = sample_play.get("playEvents", [])
                logger.info(f"Total measurement frames (pitches/pickoffs) in this plate appearance: {len(pitch_events)}")
                
        else:
            logger.error(f"Failed to query historical endpoint. HTTP Status Code: {response.status_code}")
            
    except requests.exceptions.RequestException as e:
        logger.error(f"Network transport level error occurred during collection sequence: {str(e)}")


# Execute extraction sequence
extract_historical_gumbo(HISTORICAL_GAME_PK)

2026-06-15 10:03:21 [INFO] Requesting historical GUMBO payload for gamePk: 717325
2026-06-15 10:03:21 [INFO] --- GAME METADATA RECOVERY ---
2026-06-15 10:03:21 [INFO] Matchup: St. Louis Cardinals vs Chicago Cubs
2026-06-15 10:03:21 [INFO] Final Score: St. Louis Cardinals 3 - Chicago Cubs 4
2026-06-15 10:03:21 [INFO] Total Innings Tracked: 9th
2026-06-15 10:03:21 [INFO] Successfully processed 75 complete plate appearances (at-bats).
2026-06-15 10:03:21 [INFO] --- SAMPLE AT-BAT DATA EXTRACTION ---
2026-06-15 10:03:21 [INFO] Inning: TOP 1
2026-06-15 10:03:21 [INFO] Event: Strikeout | Description: Dylan Carlson strikes out swinging.
2026-06-15 10:03:21 [INFO] Final At-Bat Count: 1B / 3S
2026-06-15 10:03:21 [INFO] Total measurement frames (pitches/pickoffs) in this plate appearance: 7


In [7]:
import logging
import sys
import boto3
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.fs as pafs

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("pyarrow_session_verifier")

# Target S3 configuration
BUCKET_NAME = "mlb-265753586044-us-east-1-an"
BASE_PREFIX = "data"

# Expected schemas and validation targets mapping
EXPECTED_TABLES = {
    "PITCHES": {
        "path": f"{BASE_PREFIX}/season=2024/",
        "file_template": "pitches_batch_",
        "primary_key": ["game_pk", "play_index", "pitch_sequence_index"],
        "sample_cols": {
            "game_pk": "int64",
            "season": "int64",
            "game_date": "string",
            "is_pitch": "bool",
            "release_speed": "double",
            "batter_id": "int64",
            "pitcher_id": "int64",
        },
    },
    "BOXSCORE_BATTING": {
        "path": f"{BASE_PREFIX}/season=2024/",
        "file_template": "boxscore_batting_batch_",
        "primary_key": ["game_pk", "player_id", "side"],
        "sample_cols": {
            "game_pk": "int64",
            "player_id": "int64",
            "side": "string",
            "game_ab": "int64",
            "season_ops": "double",
        },
    },
    "BOXSCORE_PITCHING": {
        "path": f"{BASE_PREFIX}/season=2024/",
        "file_template": "boxscore_pitching_batch_",
        "primary_key": ["game_pk", "player_id", "side"],
        "sample_cols": {
            "game_pk": "int64",
            "player_id": "int64",
            "side": "string",
            "game_innings_pitched": "double",
            "season_era": "double",
        },
    },
    "RUNNERS": {
        "path": f"{BASE_PREFIX}/season=2024/",
        "file_template": "runners_batch_",
        "primary_key": ["game_pk", "play_index", "runner_id"],
        "sample_cols": {
            "game_pk": "int64",
            "play_index": "int64",
            "runner_id": "int64",
            "is_out": "bool",
        },
    },
    "LINESCORE": {
        "path": f"{BASE_PREFIX}/season=2024/",
        "file_template": "linescore_batch_",
        "primary_key": ["game_pk", "inning"],
        "sample_cols": {
            "game_pk": "int64",
            "inning": "int64",
            "home_runs": "int64",
            "away_runs": "int64",
        },
    },
    "HITS": {
        "path": f"{BASE_PREFIX}/season=2024/",
        "file_template": "hits_batch_",
        "primary_key": None,
        "sample_cols": {
            "game_pk": "int64",
            "inning": "int64",
            "batter_id": "int64",
            "pitcher_id": "int64",
            "hit_x": "double",
        },
    },
    "PLAYERS": {
        "path": f"{BASE_PREFIX}/players/",
        "file_template": "players_batch_",
        "primary_key": ["player_id"],
        "sample_cols": {
            "player_id": "int64",
            "full_name": "string",
            "position_code": "string",
            "is_active": "bool",
        },
    },
}


def verify_schemas_with_explicit_session():
    """Validates structural metadata using boto3 resolved active session credentials passed to PyArrow."""
    try:
        # Resolve active credentials using the underlying boto3 credential engine
        session = boto3.Session()
        credentials = session.get_credentials().get_frozen_credentials()
        
        if not credentials:
            logger.error("No active AWS credentials found. Run your authorization sequence again.")
            sys.exit(1)

        # Inject session token, access keys, and region explicitly into PyArrow FileSystem
        s3_fs = pafs.S3FileSystem(
            access_key=credentials.access_key,
            secret_key=credentials.secret_key,
            session_token=credentials.token,
            region="us-east-1"
        )
    except Exception as e:
        logger.error(f"Failed to extract session or initialize PyArrow S3FileSystem: {e}")
        sys.exit(1)

    all_passed = True
    verification_summary = {}
    loaded_datasets = {}

    print("\n" + "=" * 80)
    print("MLB RAW DATA SCHEMA INTEGRITY VERIFICATION REPORT (SESSION RESOLVED)")
    print("=" * 80 + "\n")

    for table_name, meta in EXPECTED_TABLES.items():
        logger.info(f"Analyzing Table via PyArrow Dataset: {table_name}")
        verification_summary[table_name] = {"status": "PASSED", "failures": []}

        try:
            full_dir_path = f"{BUCKET_NAME}/{meta['path']}"
            selector = pafs.FileSelector(full_dir_path, recursive=True)
            files = s3_fs.get_file_info(selector)
            
            target_files = [
                f.path for f in files 
                if meta["file_template"] in f.path and f.path.endswith(".parquet")
            ]

            if not target_files:
                msg = f"No parquet files discovered matching prefix: {meta['file_template']}"
                verification_summary[table_name]["failures"].append(msg)
                verification_summary[table_name]["status"] = "FAILED"
                continue

            # Load read-only structural metadata
            dataset = ds.dataset(target_files, filesystem=s3_fs, format="parquet")
            loaded_datasets[table_name] = dataset
            pa_schema = dataset.schema

            # --- Check 1: Structure & Column Type Verification ---
            for target_col, expected_type in meta["sample_cols"].items():
                try:
                    field_idx = pa_schema.get_field_index(target_col)
                    if field_idx == -1:
                        msg = f"Missing core column: '{target_col}'"
                        verification_summary[table_name]["failures"].append(msg)
                        verification_summary[table_name]["status"] = "FAILED"
                    else:
                        actual_field = pa_schema.field(field_idx)
                        actual_type_str = str(actual_field.type)
                        
                        if expected_type == "string" and "string" in actual_type_str:
                            continue
                        if expected_type not in actual_type_str:
                            msg = f"Type mismatch for '{target_col}'. Expected base: {expected_type}, Found: {actual_type_str}"
                            verification_summary[table_name]["failures"].append(msg)
                            verification_summary[table_name]["status"] = "FAILED"
                except Exception as ex:
                    verification_summary[table_name]["failures"].append(f"Column processing error: {ex}")
                    verification_summary[table_name]["status"] = "FAILED"

            # --- Check 2: Primary Key Constraints Uniqueness ---
            if meta["primary_key"] and verification_summary[table_name]["status"] == "PASSED":
                # Project columns and load to dataframe
                pk_table = dataset.to_table(columns=meta["primary_key"])
                pk_df = pk_table.to_pandas()
                
                # Calculate duplicates of the RAW rows loaded from all files
                raw_duplicate_count = pk_df.duplicated().sum()
                
                # DISTINCT check: see if the keys are functionally unique once file-level duplication is ignored
                total_rows = len(pk_df)
                unique_keys = pk_df.groupby(meta["primary_key"]).size().shape[0]
                
                if unique_keys != (total_rows - raw_duplicate_count):
                    # This would mean that even within deduplicated data, the primary key fails
                    msg = f"True Schema Error: Key combination {meta['primary_key']} is non-unique even after batch deduplication."
                    verification_summary[table_name]["failures"].append(msg)
                    verification_summary[table_name]["status"] = "FAILED"
                elif raw_duplicate_count > 0:
                    logger.warning(
                        f"[{table_name}] Contains {raw_duplicate_count} duplicate rows across batch files, but the structural schema definition is valid."
                    )

        except Exception as e:
            msg = f"Access or structural discovery failure: {str(e)}"
            verification_summary[table_name]["failures"].append(msg)
            verification_summary[table_name]["status"] = "FAILED"

    # --- Check 3: Relational Cross-Table Foreign Key Validation ---
    print("-" * 80)
    print("CROSS-TABLE RELATIONSHIP CHECKS")
    print("-" * 80)

    if (
        verification_summary.get("PITCHES", {}).get("status") == "PASSED"
        and verification_summary.get("PLAYERS", {}).get("status") == "PASSED"
    ):
        try:
            pitches_batters = set(
                loaded_datasets["PITCHES"].to_table(columns=["batter_id"]).to_pandas()["batter_id"].unique()
            )
            players_ids = set(
                loaded_datasets["PLAYERS"].to_table(columns=["player_id"]).to_pandas()["player_id"].unique()
            )

            pitches_batters.discard(-1)
            orphans = pitches_batters.difference(players_ids)
            if orphans:
                print(
                    f"[WARNING] Found {len(orphans)} unique 'batter_id' entries in PITCHES missing from PLAYERS dictionary."
                )
            else:
                print(
                    "[OK] Referential Integrity: All batters mapped to active player bio records."
                )
        except Exception as e:
            print(f"[ERROR] Relationship validation failed: {e}")

    # --- Final Summary Presentation ---
    print("\n" + "=" * 80)
    print("FINAL SCHEMA VERIFICATION RESULTS")
    print("=" * 80)
    for table, results in verification_summary.items():
        status_str = f"[{results['status']}]"
        print(f"{table:<20} {status_str}")
        if results["failures"]:
            all_passed = False
            for failure in results["failures"]:
                print(f"  └── Error: {failure}")

    print("=" * 80)
    if all_passed:
        print("SUCCESS: Parquet structural rules strictly match assertions in SCHEMA.md.")
        sys.exit(0)
    else:
        print("CRITICAL: Schema drift or rule mismatches identified between S3 and metadata definitions.")
        sys.exit(1)


if __name__ == "__main__":
    verify_schemas_with_explicit_session()

2026-06-25 16:36:22,656 - INFO - Analyzing Table via PyArrow Dataset: PITCHES



MLB RAW DATA SCHEMA INTEGRITY VERIFICATION REPORT (SESSION RESOLVED)



2026-06-25 16:36:32,886 - WARNING - [PITCHES] Contains 12952 duplicate rows across batch files, but the structural schema definition is valid.
2026-06-25 16:36:32,887 - INFO - Analyzing Table via PyArrow Dataset: BOXSCORE_BATTING
2026-06-25 16:36:38,308 - WARNING - [BOXSCORE_BATTING] Contains 806 duplicate rows across batch files, but the structural schema definition is valid.
2026-06-25 16:36:38,308 - INFO - Analyzing Table via PyArrow Dataset: BOXSCORE_PITCHING
2026-06-25 16:36:43,770 - WARNING - [BOXSCORE_PITCHING] Contains 326 duplicate rows across batch files, but the structural schema definition is valid.
2026-06-25 16:36:43,771 - INFO - Analyzing Table via PyArrow Dataset: RUNNERS
2026-06-25 16:36:49,806 - WARNING - [RUNNERS] Contains 15685 duplicate rows across batch files, but the structural schema definition is valid.
2026-06-25 16:36:49,807 - INFO - Analyzing Table via PyArrow Dataset: LINESCORE
2026-06-25 16:36:55,434 - WARNING - [LINESCORE] Contains 355 duplicate rows acro

--------------------------------------------------------------------------------
CROSS-TABLE RELATIONSHIP CHECKS
--------------------------------------------------------------------------------
[OK] Referential Integrity: All batters mapped to active player bio records.

FINAL SCHEMA VERIFICATION RESULTS
PITCHES              [PASSED]
BOXSCORE_BATTING     [PASSED]
BOXSCORE_PITCHING    [PASSED]
RUNNERS              [PASSED]
LINESCORE            [PASSED]
HITS                 [PASSED]
PLAYERS              [PASSED]
SUCCESS: Parquet structural rules strictly match assertions in SCHEMA.md.


SystemExit: 0

/opt/miniconda3/envs/pred/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
